# Dynamic time warping: the UCR archive baselines

Is a heartbeat normal? Is a major earthquake about to follow? Since 2002 the UCR Time Series Archive has been where new time series classifiers are tested, and every dataset in it comes with three baselines: the error rates of the 1-nearest-neighbour classifier with the Euclidean distance (ED), with dynamic time warping (DTW) constrained by a warping window learned from the training set, and with unconstrained DTW. Dau, Bagnall, Keogh and co-authors (2019) describe how the baselines are computed, how a learned window can fail, how picking datasets can make a worthless idea look good, and how to test a new method honestly. Their baselines and their Table T are printed beside Julia's, for three datasets of the archive.

The data, from PollisDatasets: `ecg200`, two classes of heartbeats; `twoleadecg`, the two leads of an electrocardiogram; and `earthquakes`, whether a major earthquake follows a reading. Each is used with the archive's own train/test split. The DTW distance comes from DynamicAxisWarping.jl, with the warping window, the distance and the choice of window of the authors' MATLAB code (`one_NN.m` and `run_baseline_full_serial.m`, on the paper's supporting web page).

In [ ]:
Pollis.packages("DynamicAxisWarping", "Distances", "PollisDatasets", "Plots")

## Set-up

`series` turns a PollisDatasets table, one row per series with its split and class, into a training and a test set, each a matrix with one series per column and its labels; `zscore` normalises each series to mean 0 and standard deviation 1, as `one_NN.m` does before classifying; and `replicate` prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using DynamicAxisWarping, Distances, PollisDatasets, Plots, Printf, Statistics, Tables

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

"""
	series(table)

The training and test sets of a PollisDatasets table with columns `split`, `label` and then the values: each a matrix `X` with one series per column, and the labels `y`.
"""
function series(table)
	t = Tables.columntable(table)
	X = permutedims(reduce(hcat, [Float64.(t[k]) for k in keys(t)[3:end]]))  # one column per series
	train = t.split .== "train"
	(train=(X=X[:, train], y=t.label[train]), test=(X=X[:, .!train], y=t.label[.!train]))
end

zscore(X) = (X .- mean(X, dims=1)) ./ std(X, dims=1)  # each series (column), as MATLAB's zscore
normalised(d) = (train=(X=zscore(d.train.X), y=d.train.y), test=(X=zscore(d.test.X), y=d.test.y))

data = ["ECG200" => normalised(series(dataset("ecg200"))),
	"TwoLeadECG" => normalised(series(dataset("twoleadecg"))),
	"Earthquakes" => normalised(series(dataset("earthquakes")))];

## 1. The data

ECG200 is a set of heartbeats, each 96 readings of one electrocardiogram lead, labelled normal or myocardial infarction; TwoLeadECG, 82 readings of one record of the MIT-BIH Long-Term ECG Database, the task being to tell which of its two leads, signal 0 or signal 1, a series comes from; Earthquakes, 512 hourly readings of a seismic sensor, labelled by whether a major event followed. The default rate is the test error of always predicting the largest class of the test set, as the authors' `compute_default_error_rate.m` computes it: a classifier is only useful below it. The training sets are small, 23 series for TwoLeadECG, and smaller than the test sets, on purpose: "labeled train data are usually expensive" (p. 8).

In [ ]:
defaultrate(y) = 1 - maximum(count(==(c), y) for c in unique(y)) / length(y)
@printf("%-12s%7s%7s%8s   %s\n", "", "train", "test", "length", "series per class (train + test)")
for (name, d) in data
	y = [d.train.y; d.test.y]
	@printf("%-12s%7d%7d%8d   %s\n", name, length(d.train.y), length(d.test.y), size(d.train.X, 1), join(["$c: $(count(==(c), y))" for c in sort(unique(y))], ", "))
end
replicate("Default rates (DataSummary.csv, the archive's web page)", [("$name", defaultrate(d.test.y), p) for ((name, d), p) in zip(data, ["0.3600", "0.4996", "0.2518"])])

plot([begin
		p = plot(title=name, titlefontsize=9, legend=:topright, legendfontsize=7, xlabel="t")
		for (k, c) in enumerate(sort(unique(d.train.y)))
			columns = findall(==(c), d.train.y)[1:3]  # the first three training series of each class
			plot!(p, d.train.X[:, columns], color=k, label=["class $c" "" ""], lw=0.8)
		end
		p
	end for (name, d) in data]..., layout=(3, 1), size=(700, 650))

## 2. The three baselines

The 1-nearest-neighbour classifier gives each test series the class of its closest training series. With the Euclidean distance, value i is matched with value i. DTW lets value i be matched with value j, with |i - j| at most the warping window, and finds the matching with the smallest sum of squared differences; with a window of 0 it is the squared Euclidean distance, which ranks the series as the Euclidean distance does. As in `one_NN.m`, a window of w per cent is `ceil(length * w / 100)` points (`cld` below), and DTW with w = 100 is unconstrained.

The learned window is chosen on the training set alone: for w = 0, 1, ..., 10 per cent (`max_w = 0.1` in the authors' `main.m`), the leave-one-out error, each training series classified by its nearest neighbour among the others, and the smallest w with the smallest error is kept ("choosing the smallest value of w that minimizes the average train error rate", p. 2). `errors` computes both error rates for one window; Earthquakes takes a few minutes, since each window needs about 100,000 DTW distances.

In [ ]:
"""
	errors(d, r; loo = true)

The test error of the 1-nearest-neighbour classifier with DTW of window `r` points (r = 0: Euclidean), and its leave-one-out error on the training set.
"""
function errors(d, r; loo=true)
	cost(a, b) = dtw_cost(a, b, SqEuclidean(), r)
	X, y = d.train.X, d.train.y
	n = size(X, 2)
	looerror = NaN
	if loo
		D = fill(Inf, n, n)  # a series is not its own neighbour
		for i in 1:n, j in i+1:n
			D[i, j] = D[j, i] = cost(view(X, :, i), view(X, :, j))
		end
		looerror = mean(y[argmin(D[:, i])] != y[i] for i in 1:n)
	end
	test = mean(y[argmin([cost(view(X, :, i), x) for i in 1:n])] != c for (x, c) in zip(eachcol(d.test.X), d.test.y))
	(loo=looerror, test=test)
end

windows = 0:10  # per cent of the length
results = Dict(name => (curve=[errors(d, cld(w * size(d.train.X, 1), 100)) for w in windows],
	unconstrained=errors(d, size(d.train.X, 1); loo=false).test) for (name, d) in data)
learned(name) = windows[argmin([e.loo for e in results[name].curve])]  # the first, so the smallest, minimum

published = Dict("ECG200" => ("0.1200", "0.1200", "0", "0.2300"), "TwoLeadECG" => ("0.2529", "0.1317", "4", "0.0957"),
	"Earthquakes" => ("0.2878", "0.2734", "6", "0.2806"))
for (name, _) in data
	r, (ed, dtw, w, full) = results[name], published[name]
	replicate("$name (DataSummary.csv)", [("ED", r.curve[1].test, ed), ("DTW, learned window", r.curve[learned(name) + 1].test, dtw),
		("learned window, per cent", learned(name), w), ("DTW, unconstrained", r.unconstrained, full)])
end

## 3. Does the learned window predict the test error? Fig. 2

Learning the window "works well in practice. However, it can produce poor results as in some situations, the best w in training may not be the best w for testing" (p. 2), which "only occurs for very small training sets". Fig. 2 draws the leave-one-out and test errors against the window for six other datasets; below, the same curves for these three. With 23 training series, TwoLeadECG's leave-one-out error moves in steps of 1/23, about 0.043, and flattens from the learned window on, while the test error goes on falling; Earthquakes' test error stays above its default rate for every window.

In [ ]:
plot([begin
		r = results[name]
		p = plot(windows, [e.loo for e in r.curve], label="leave-one-out (train)", marker=:circle, ms=2.5, title=name, titlefontsize=9,
			xlabel="warping window, per cent", ylabel="error rate", legendfontsize=7)
		plot!(p, windows, [e.test for e in r.curve], label="test", marker=:square, ms=2.5, lw=2)
		vline!(p, [learned(name)], label="learned window", linestyle=:dash)
		hline!(p, [defaultrate(d.test.y)], label="default rate", linestyle=:dot)
	end for (name, d) in data]..., layout=(1, 3), size=(950, 330), bottom_margin=5Plots.mm, left_margin=4Plots.mm)

In [ ]:
@printf("%-12s%10s%14s%12s%14s\n", "", "learned w", "test error", "best w", "test error")
for (name, _) in data
	test = [e.test for e in results[name].curve]
	@printf("%-12s%10d%14.4f%12d%14.4f\n", name, learned(name), test[learned(name) + 1], windows[argmin(test)], minimum(test))
end

The best window in the last two columns is chosen by looking at the test set, so it is not a result anyone could claim: it only shows how much was lost by learning the window from the training set.

## 4. The Texas Sharpshooter plot: Fig. 5

A method that wins on some datasets and loses on others is only useful if one can tell in advance which datasets it wins on. The Texas Sharpshooter plot (Batista et al. 2014) puts each dataset at its expected gain, the ratio of the two methods' accuracies estimated on the training set alone, here the leave-one-out accuracies, against its actual gain, the same ratio on the test set (p. 8). Datasets in the top right were predicted to gain and did; in the bottom right, the training set promised a gain the test set did not deliver. Fig. 5 shows it for ED against DTW with the learned window on all 128 datasets; below, the same for these three. Constrained DTW includes ED as the window 0, so the expected gain is never below 1.

In [ ]:
gain(dtw, ed) = (1 - dtw) / (1 - ed)  # ratio of accuracies
points = [(name, gain(results[name].curve[learned(name) + 1].loo, results[name].curve[1].loo),
	gain(results[name].curve[learned(name) + 1].test, results[name].curve[1].test)) for (name, _) in data]
foreach(p -> @printf("%-12s expected gain %.4f, actual gain %.4f\n", p...), points)
scatter([p[2] for p in points], [p[3] for p in points], series_annotations=text.([p[1] for p in points], 8, :left), label="",
	xlabel="expected accuracy gain (train)", ylabel="actual accuracy gain (test)", xlims=(0.9, 1.4), ylims=(0.9, 1.4), size=(450, 420))
vline!([1], color=:gray, label=""); hline!([1], color=:gray, label="")

## 5. Cherry picking: Table T

Section IV makes up a method, FQT, which deletes the first quarter of every series and uses the Euclidean distance, "obviously not a clever thing to do", and reports it on the 20 datasets of the old archive where it happens to beat ED (Table T, p. 4): with "all the numbers presented in the table and figure ... true values". On all 85 datasets FQT wins on 19 and loses on 66 (p. 4). All three of these datasets are in Table T. As in the authors' `UCR_throw_data.m`, a quarter is `round((length + 1) * 0.25)` values, and the shortened series are normalised again.

Earthquakes is one of the 14 datasets whose training and test sets were swapped by mistake in the 2015 archive and put back for the 2018 release (Table I, p. 9); Table T, computed on the old archive, uses the swapped sets.

In [ ]:
quarter(X) = X[round(Int, (size(X, 1) + 1) * 0.25)+1:end, :]  # deletes the first quarter
fqt(d) = normalised((train=(X=quarter(d.train.X), y=d.train.y), test=(X=quarter(d.test.X), y=d.test.y)))
swapped(d) = (train=d.test, test=d.train)  # the 2015 split
ed(d) = errors(d, 0; loo=false).test

sets = Dict(data)
old = swapped(sets["Earthquakes"])
replicate("Table T (p. 4): ED and FQT error rates", [
	("ECG200, ED", ed(sets["ECG200"]), "0.120"), ("ECG200, FQT", ed(fqt(sets["ECG200"])), "0.110"),
	("TwoLeadECG, ED", ed(sets["TwoLeadECG"]), "0.253"), ("TwoLeadECG, FQT", ed(fqt(sets["TwoLeadECG"])), "0.241"),
	("Earthquakes 2015, ED", ed(old), "0.326"), ("Earthquakes 2015, FQT", ed(fqt(old)), "0.301")])
@printf("\nEarthquakes, 2018 split: ED %.4f, FQT %.4f\n", ed(sets["Earthquakes"]), ed(fqt(sets["Earthquakes"])))

On the archive as it now is, FQT loses on Earthquakes: the win in Table T came from a split that was a mistake. A result that rests on a handful of datasets can turn on such details, which is why the paper asks every new method to be tested on all the datasets, or to say why not (the checklist, p. 7).

## 6. The branch: warp or not?

Whether to warp is decided on the training set alone, as the Texas Sharpshooter plot asks: DTW with the learned window is kept for a dataset if its leave-one-out error is below ED's, and ED otherwise. `warping` gives the decision for each dataset; the test set then says whether it was right, and whether the method kept beats the default rate at all. For Earthquakes, no 1-nearest-neighbour classifier, warped or not, does better than always predicting no earthquake.

In [ ]:
warping = Dict(name => results[name].curve[learned(name) + 1].loo < results[name].curve[1].loo for (name, _) in data)
@printf("%-12s%-26s%12s%12s%14s\n", "", "kept (train only)", "its test", "the other", "default rate")
for (name, d) in data
	r = results[name]
	kept, other = warping[name] ? (r.curve[learned(name) + 1].test, r.curve[1].test) : (r.curve[1].test, r.curve[learned(name) + 1].test)
	@printf("%-12s%-26s%12.4f%12.4f%14.4f\n", name, warping[name] ? "DTW, window $(learned(name)) per cent" : "ED", kept, other, defaultrate(d.test.y))
end
warping

## 7. What differs, and why

All 21 published values agree to every printed digit: the 3 default rates; ED, DTW with the learned window, the learned window and unconstrained DTW for each dataset (DataSummary.csv); and the 6 error rates of Table T. Small points:

- **The published baselines are in DataSummary.csv, not in the paper.** The paper describes them (Section II) and points to the supporting web page and the archive's page for the numbers. Table T's error rates are in the paper itself, to 3 decimals.
- **The window rule is the authors' code.** `one_NN.m` converts a window of w per cent with `ceil`: rounding instead gives TwoLeadECG a learned-window error of 0.1449, not 0.1317.
- **Normalised twice.** The archive's series are already normalised; `one_NN.m` normalises them again, which changes nothing but the last bits.
- **Earthquakes in Table T** uses the 2015 split (section 5); the DataSummary.csv values use the 2018 one.
- **Not replicated:** Fig. 2 and Fig. 5 draw other datasets, or all 128, without printing their values, and the critical-difference comparison (Section V) needs every dataset and the other methods.

## References

- Bagnall, A., Lines, J., Bostrom, A., Large, J. and Keogh, E. (2017). The great time series classification bake off: a review and experimental evaluation of recent algorithmic advances. *Data Mining and Knowledge Discovery* 31(3), 606-660.
- Batista, G. E., Keogh, E. J., Tataw, O. M. and De Souza, V. M. A. (2014). CID: an efficient complexity-invariant distance for time series. *Data Mining and Knowledge Discovery* 28(3), 634-669.
- Dau, H. A., Silva, D. F., Petitjean, F., Forestier, G., Bagnall, A., Mueen, A. and Keogh, E. (2018). Optimizing dynamic time warping's window width for time series data mining applications. *Data Mining and Knowledge Discovery*.
- Dau, H. A., Bagnall, A., Kamgar, K., Yeh, C.-C. M., Zhu, Y., Gharghabi, S., Ratanamahatana, C. A. and Keogh, E. (2019). The UCR time series archive. *IEEE/CAA Journal of Automatica Sinica* 6(6), 1293-1305. doi:10.1109/jas.2019.1911747. Pages and tables are those of the open version, arXiv:1810.07758v2.
- Data: ECG200, TwoLeadECG and Earthquakes from PollisDatasets (`ecg200`, `twoleadecg`, `earthquakes`), the archive's copies at timeseriesclassification.com. Published baselines: DataSummary.csv, https://www.cs.ucr.edu/~eamonn/time_series_data_2018/. Code: the authors' MATLAB code, https://www.cs.ucr.edu/~hdau001/ucr_archive/.